In [ ]:
!pip install scikit-learn scipy torch pandas nltk

# Dimensionality reduction with singular value decomposition for image compression

The following code is adapted from: https://www.geeksforgeeks.org/data-science/singular-value-decomposition-svd/

We will use the svd implementation from scipy to compute the three matrices (U, S, V)

In [ ]:
import cv2
import matplotlib.pyplot as plt
import numpy as np
from scipy.linalg import svd

cat = cv2.imread("data/cat.png")
plt.imshow(cat)
print(cat.shape)

cat_flat = np.reshape(cat, (cat.shape[0], cat.shape[1]*cat.shape[2]))/255
print(cat_flat.shape)
orig_size=int(cat_flat.shape[0]*cat_flat.shape[1]/1024)
print(f"{orig_size}KB")

U, S, V_T = svd(cat_flat, full_matrices=False)
S = np.diag(S)
fig, ax = plt.subplots(5, 3, figsize=(10, 20))

curr_fig = 0
for r in [5, 10, 15, 30, 100]:
    cat_approx = U[:, :r] @ S[0:r, :r] @ V_T[:r, :]

    sz=((V_T[:r, :]).shape[0]*(V_T[:r, :]).shape[1] + (S[0:r, :r]).shape[0]*S[0:r, :r].shape[1]+ V_T[:r, :].shape[0]*V_T[:r, :].shape[1])/1024
    cat_approx = np.reshape(cat_approx, (cat_approx.shape[0], cat_approx.shape[1]//3, 3))
    error=np.sqrt((cat_approx-cat/255)**2)
    ax[curr_fig][0].imshow(cat_approx)
    ax[curr_fig][0].set_title("k = " + str(r)+ f" size={int(sz)}KB")
    ax[curr_fig, 0].axis('off')
    ax[curr_fig][1].set_title(f"Original Image size={int(orig_size)}KB")
    ax[curr_fig][1].imshow(cat)
    ax[curr_fig, 1].axis('off')
    ax[curr_fig][2].set_title(f"Error")
    ax[curr_fig][2].imshow(error)
    ax[curr_fig, 2].axis('off')
    
    curr_fig += 1
plt.show()

# Moving on to words

We will perfom the same thing, but for words.

**Step 1**: We start by loading the 20newsgroups dataset using sklearn. Also, we **filter out common words** using the list of stopwords from NLTK

In [ ]:
from sklearn.datasets import fetch_20newsgroups
newsgroups_train = fetch_20newsgroups(subset='train')

In [ ]:
# remove stop-words

import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize

nltk.download('stopwords')
nltk.download('punkt_tab')

# Sample text
text = "This is a sample sentence showing stopword removal."

# Get English stopwords and tokenize
stop_words = set(stopwords.words('english'))
filtered_data=[]
for example in newsgroups_train.data:
    tokens = word_tokenize(example.lower())
    filtered_tokens = [word for word in tokens if word not in stop_words]
    filtered_data.append(' '.join(filtered_tokens))



**Step 2**: We now build the term document matrix. For each word, we mark the documents in which it appears.

In [ ]:
from sklearn.feature_extraction.text import CountVectorizer
import pandas as pd

# Count Vectorizer
vect = CountVectorizer()
vects = vect.fit_transform(filtered_data)

# Select the first five rows from the data set
td = pd.DataFrame(vects.todense())
td.columns = vect.get_feature_names_out()
term_document_matrix = td.T
term_document_matrix.columns = ['Doc '+str(i) for i in range(1, len(newsgroups_train.data)+1)]
term_document_matrix['total_count'] = term_document_matrix.sum(axis=1)

# Top 1000 words 
term_document_matrix = term_document_matrix.sort_values(by ='total_count',ascending=False)[:1000].clip(0, 1)

# Print the first 10 rows 
print(term_document_matrix.drop(columns=['total_count']).head(10))

**Step 3**: We compute the singula value decomposition 

In [ ]:
U, S, V_T = svd(term_document_matrix, full_matrices=False)
S = np.diag(S)
keep_dims=100

**Step 4**: We calculate low-dimensional projection using the top `keep_dims`, based on their variance

In [ ]:
V=V_T.transpose()
#term_document_matrix=np.array(term_document_matrix)
term_document_reduced=np.array(term_document_matrix)@V[:,:keep_dims]
print(term_document_reduced)

**Step 5**: Visualization

In [ ]:
word2index={}
feats=term_document_matrix.index
for w in feats:
    word2index[w]=len(word2index)

In [ ]:
n_words=50

x=term_document_reduced[:n_words,0]
y=term_document_reduced[:n_words,1]
labels=term_document_matrix.index[:n_words]
plt.rcParams["figure.figsize"] = (12,12)
plt.scatter(x, y, label=labels)
for i, txt in enumerate(labels):
    plt.annotate(txt, (x[i], y[i]), xytext=(5, 5), textcoords='offset points')
plt.show()

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
cosine_similarity(X=term_document_reduced[word2index['organization']].reshape(1,100), Y=term_document_reduced[word2index['university']].reshape(1,100))

In [ ]:
cosine_similarity(X=term_document_reduced[word2index['organization']].reshape(1,100), Y=term_document_reduced[word2index['apr']].reshape(1,100))

In [ ]:
cosine_similarity(X=term_document_reduced[word2index['bad']].reshape(1,100), Y=term_document_reduced[word2index['apr']].reshape(1,100))

In [ ]:
cosine_similarity(X=term_document_reduced[word2index['write']].reshape(1,100), Y=term_document_reduced[word2index['read']].reshape(1,100))